# Large Run Step 7: Dashboard Driver

Purpose: prepare or launch dashboards using already materialized dashboard datasets and QC tables. The notebook prints commands and previews small samples only.

Outputs: dashboard metric dataset, dashboard summaries, and launch commands for metrics and QC dashboards.


## Setup
Purpose: load config/output paths and shared settings.

Outputs: printed paths and helper variables.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import time

import pandas as pd

from spatial_vtk.config import SpatialVTKConfig
from spatial_vtk.config.outputs import resolve_output_path
from spatial_vtk.io import load_output_table, preview_output_table, write_output_table


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "spatial_vtk").exists():
            return candidate
    return start


repo_root = find_repo_root()
config_candidates = [
    repo_root / "runs" / "spatial_vtk_config.yaml",
    Path.cwd() / "spatial_vtk_config.yaml",
    Path.cwd() / "runs" / "spatial_vtk_config.yaml",
]
config_path = next((path for path in config_candidates if path.exists()), config_candidates[0])
cfg = SpatialVTKConfig.from_file(config_path).activate()

outputs_root = Path(cfg.path("outputs.root", create_parent=True) or (cfg.root_dir / "runs" / "outputs"))
tables_dir = Path(cfg.path("outputs.tables", create_parent=True) or (outputs_root / "tables"))
figures_dir = Path(cfg.path("outputs.figures", create_parent=True) or (outputs_root / "figures"))
slurm_dir = outputs_root / "slurm"
logs_dir = outputs_root / "logs"
for directory in (tables_dir, figures_dir, slurm_dir, logs_dir):
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOCAL = os.environ.get("SVTK_RUN_LOCAL", "0") == "1"
SUBMIT_SLURM = os.environ.get("SVTK_SUBMIT_SLURM", "0") == "1"
OVERWRITE = os.environ.get("SVTK_OVERWRITE", "0") == "1"
QC_CHUNKSIZE = int(os.environ.get("SVTK_QC_CHUNKSIZE", "1000000"))
PREVIEW_ROWS = int(os.environ.get("SVTK_PREVIEW_ROWS", "5"))

print(f"repo_root: {repo_root}")
print(f"config_path: {config_path}")
print(f"outputs_root: {outputs_root}")
print(f"tables_dir: {tables_dir}")
print(f"figures_dir: {figures_dir}")
print(f"SUBMIT_SLURM={SUBMIT_SLURM} RUN_LOCAL={RUN_LOCAL} OVERWRITE={OVERWRITE}")


## Helpers
Purpose: define skip/status helpers.

Outputs: helper functions only.


In [ ]:
def exists(path):
    return Path(path).exists()


def should_run(*paths):
    return OVERWRITE or not all(Path(path).exists() for path in paths)


def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"path": str(path), "exists": False, "size_gb": None, "modified": None}
    return {
        "path": str(path),
        "exists": True,
        "size_gb": round(path.stat().st_size / 1024**3, 3),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }


def show_files(paths):
    display(pd.DataFrame([file_info(path) for path in paths]))


def submit_script(script_path, *, submit=SUBMIT_SLURM):
    script_path = Path(script_path)
    print(f"script: {script_path}")
    if submit:
        result = subprocess.run(["sbatch", str(script_path)], text=True, capture_output=True, check=False)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"sbatch failed with return code {result.returncode}")
    else:
        print(f"Set SVTK_SUBMIT_SLURM=1 and rerun this cell, or submit manually:")
        print(f"sbatch {shlex.quote(str(script_path))}")


def write_python_slurm_script(script_name, python_body, *, job_name, walltime="24:00:00", memory="32G", cpus=1):
    script_path = slurm_dir / script_name
    body = textwrap.dedent(python_body).strip()
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={logs_dir}/{job_name}_%j.out
#SBATCH --error={logs_dir}/{job_name}_%j.err
#SBATCH --time={walltime}
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}

set -euo pipefail
cd {repo_root}
source /project2/jvidale_1700/miniforge3/etc/profile.d/conda.sh || true
conda activate spatial-vtk-py312 || true
python - <<'PYJOB'
{body}
PYJOB
"""
    script_path.write_text(script, encoding="utf-8")
    script_path.chmod(0o755)
    return script_path


def preview_table_path(path, n=PREVIEW_ROWS, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq
        parquet = pq.ParquetFile(path)
        available = parquet.schema.names
        selected = [column for column in (columns or available) if column in available]
        for batch in parquet.iter_batches(batch_size=max(int(n), 1), columns=selected):
            return batch.to_pandas().head(n)
        return pd.DataFrame(columns=selected)
    return pd.read_csv(path, nrows=n, usecols=columns)


## Resolve Dashboard Inputs
Purpose: check dashboard datasets, metric outputs, and QC inventory paths.

Outputs: status table only.


In [ ]:
metrics_long_path = resolve_output_path("metrics_long", kind="table", create_parent=True)
metrics_dashboard_root = tables_dir / "dashboard_metrics"
dashboard_summary_root = tables_dir / "dashboard_summaries"
qc_inventory_path = resolve_output_path("qc_inventory", kind="table", create_parent=True)
qc_inventory_overlap_path = resolve_output_path("qc_inventory_overlap", kind="table", create_parent=True)

show_files([metrics_long_path, metrics_dashboard_root, dashboard_summary_root, qc_inventory_path, qc_inventory_overlap_path])


## Prepare Dashboard Datasets
Purpose: run dashboard preparation outside the notebook when dashboard datasets are missing.

Outputs: partitioned metrics dashboard dataset and dashboard summary files.


In [ ]:
if not metrics_long_path.exists():
    print("metrics_long.parquet is not ready yet; finish Step 3 first.")
elif should_run(metrics_dashboard_root, dashboard_summary_root):
    cmd = [
        "svtk", "metrics", "outputs",
        "--metrics", str(metrics_long_path),
        "--output-dir", str(tables_dir),
        "--format", "parquet",
        "--dashboard-partitioned",
    ]
    print(" ".join(shlex.quote(part) for part in cmd))
    if RUN_LOCAL:
        subprocess.run(cmd, check=True)
    else:
        print("Set SVTK_RUN_LOCAL=1 to prepare dashboards, or run the printed command on the login node.")
else:
    print("Dashboard datasets already exist; skipping.")


## Print Dashboard Launch Commands
Purpose: produce commands for launching dashboards without starting long-lived servers automatically.

Outputs: shell commands for metrics and QC dashboards.


In [ ]:
metrics_port = int(os.environ.get("SVTK_METRICS_DASHBOARD_PORT", "8501"))
qc_port = int(os.environ.get("SVTK_QC_DASHBOARD_PORT", "8502"))
metrics_command = " ".join([
    "svtk dashboard metrics",
    f"--metrics-root {shlex.quote(str(metrics_dashboard_root))}",
    f"--summary-root {shlex.quote(str(dashboard_summary_root))}",
    f"--port {metrics_port}",
])
qc_command = " ".join([
    "svtk dashboard qc",
    f"--trace-summary {shlex.quote(str(qc_inventory_path))}",
    f"--port {qc_port}",
])
print("Metrics dashboard command:")
print(metrics_command)
print("\nQC dashboard command:")
print(qc_command)


## Preview Dashboard Data
Purpose: inspect bounded samples only and avoid loading dashboard partitions wholesale.

Outputs: small preview from `metrics_long.parquet` if present.


In [ ]:
if metrics_long_path.exists():
    display(preview_table_path(metrics_long_path, PREVIEW_ROWS))
else:
    print("metrics_long.parquet is not ready yet.")
